# mT5 and XLM-R on FLORES-200 and the Arabic source

The sentencepiece check showed that the paper's mT5 and XLM-R counts on the translation files were
too high for 21 files, and those rows are now corrected. Two inputs to the FLORES table
(Appendix B) were not covered by that check:

1. the **Arabic source**, which is part of the 41-language FLORES matched set, and
2. the **FLORES devtest counts** themselves, computed in `04_flores_replication_and_downstream.ipynb`.

This notebook counts both with the reference `sentencepiece` library and compares them with what the
paper used. **CPU, about 5 minutes.** Upload `flores_fertility_by_language.csv` and
`results/superseded/fertility_production_v2_all11.csv` from this repository (`data/` and `results/`); outputs are collected in `results_flores_spm.zip`.

In [ ]:
!pip -q install sentencepiece
import os, json, urllib.request, subprocess
import numpy as np, pandas as pd
import sentencepiece as spm
from huggingface_hub import hf_hub_download
try:
    from google.colab import files
    need = [f for f in ["flores_fertility_by_language.csv", "fertility_production_v2_all11.csv"] if not os.path.exists(f)]
    if need:
        print("Upload:", need); files.upload()
except ImportError:
    pass
OUT = "results_flores_spm"; os.makedirs(OUT, exist_ok=True)

if not os.path.exists("flores200_dataset"):
    subprocess.run("wget -q https://dl.fbaipublicfiles.com/nllb/flores200_dataset.tar.gz && "
                   "tar xzf flores200_dataset.tar.gz", shell=True, check=True)
if not os.path.exists("quran.json"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/risan/quran-json/main/dist/quran.json", "quran.json")
ARABIC = [v["text"] for s in json.load(open("quran.json", encoding="utf-8")) for v in s["verses"]]
assert len(ARABIC) == 6236, len(ARABIC)

flores_pub = pd.read_csv("flores_fertility_by_language.csv")
corpus_pub = pd.read_csv("fertility_production_v2_all11.csv").set_index("language")
codes = sorted(flores_pub.flores_code.unique())
SENTS = {c: open(f"flores200_dataset/devtest/{c}.devtest", encoding="utf-8").read().rstrip("\n").split("\n")
         for c in codes}
print(len(codes), "FLORES codes;", len(ARABIC), "Arabic verses")

In [ ]:
MODELS = {"mt5": ("google/mt5-small", "spiece.model"), "xlmr": ("xlm-roberta-base", "sentencepiece.bpe.model")}
rows, arab = [], {}
for col, (repo, fname) in MODELS.items():
    sp = spm.SentencePieceProcessor(model_file=hf_hub_download(repo, fname))
    arab[col] = {"published": float(corpus_pub.loc["arabic_source", f"{col}_tok_per_verse"]),
                 "spm": float(np.mean([len(x) for x in sp.encode(ARABIC)]))}
    fp = flores_pub[flores_pub.tokenizer == col].set_index("flores_code").mean_tok_per_sent
    for c in codes:
        rows.append({"tokenizer": col, "flores_code": c, "published": float(fp[c]),
                     "spm": float(np.mean([len(x) for x in sp.encode(SENTS[c])]))})
    print(col, "done", flush=True)
F = pd.DataFrame(rows); F["rel_diff"] = F.spm / F.published - 1
F.to_csv(f"{OUT}/flores_counts_spm.csv", index=False)
json.dump(arab, open(f"{OUT}/arabic_source_spm.json", "w"), indent=1)

for col in MODELS:
    d = F[F.tokenizer == col]
    a = arab[col]
    print(f"\n=== {col}")
    print(f"  Arabic source: published {a['published']:.3f}  sentencepiece {a['spm']:.3f}  "
          f"({100*(a['spm']/a['published']-1):+.1f}%)")
    print(f"  FLORES devtest: max |diff| {d.rel_diff.abs().max():.2e} over {len(d)} codes")
    off = d[d.rel_diff.abs() > 1e-3]
    if len(off): print(off.round(4).to_string(index=False))
    else: print("  FLORES counts reproduce: the paper's FLORES column for this tokenizer is correct")
import shutil; shutil.make_archive("results_flores_spm", "zip", OUT)
try:
    from google.colab import files; files.download("results_flores_spm.zip")
except Exception:
    print("written results_flores_spm.zip")